# Diagnóstico: MLP vs ST-GCN

Este notebook incluye un modelo MLP simple para descartar problemas en los datos.

In [ ]:
import os, sys, json, torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, Dataset

ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path: sys.path.insert(0, str(ROOT_DIR))

from src.stgcn.stgcn_model import RealSTGCN
from src.stgcn.hand_graph import build_adjacency_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
from sklearn.preprocessing import LabelEncoder
MANIFEST = "output/train_manifest_stgcn_secuencias_fixed.csv"
df = pd.read_csv(MANIFEST)
df = df[df['quality_flag'] != 'excluded'].copy()

print("--- AUDITORÍA DE ETIQUETAS ---")
print(f"Total muestras: {len(df)}")
print(f"Muestras con etiqueta vacía: {df['label'].isna().sum()}")

df['label'] = df['label'].fillna('unknown')
le = LabelEncoder()
df['label_idx'] = le.fit_transform(df['label'])
unique_labels = le.classes_.tolist()
label_to_idx = {name: i for i, name in enumerate(unique_labels)}

print("Distribución Top 5:")
print(df['label'].value_counts().head(5))

train_df = df.sample(frac=0.8, random_state=42)
val_df = df.drop(train_df.index)

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    def forward(self, x): 
        # x: (B, C, T, V) -> Flattened
        return self.net(x), None

# Input dim = 3 (xyz) * 16 (frames) * 21 (nodes) = 1008
mlp_model = SimpleMLP(1008, len(unique_labels)).to(device)
print("Modelo MLP listo.")

In [ ]:
class STGCNDataset(Dataset):
    def __init__(self, manifest_df, label_map):
        self.df = manifest_df
        self.label_map = label_map
        self.base_dir = Path("data/processed/secuencias_stgcn")
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = Path(row['path_secuencia'])
        if not path.exists(): path = self.base_dir / path.name
        seq = np.load(path).astype(np.float32)
        # Wrist-relative
        seq = seq - seq[:, 0:1, :]
        x = torch.from_numpy(np.transpose(seq, (2, 0, 1))).float()
        y = torch.tensor(self.label_map[str(row['label'])], dtype=torch.long)
        return x, y

train_loader = DataLoader(STGCNDataset(train_df, label_to_idx), batch_size=128, shuffle=True)
val_loader = DataLoader(STGCNDataset(val_df, label_to_idx), batch_size=128, shuffle=False)

optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("Entrenando MLP (Diagnóstico)...")
for epoch in range(15):
    mlp_model.train()
    correct = 0; total = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs, _ = mlp_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        _, pred = torch.max(outputs, 1)
        total += labels.size(0); correct += (pred == labels).sum().item()
    
    mlp_model.eval()
    v_correct = 0; v_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs, _ = mlp_model(inputs)
            _, pred = torch.max(outputs, 1)
            v_total += labels.size(0); v_correct += (pred == labels).sum().item()
    
    print(f"Epoch {epoch+1} | Train: {100*correct/total:.2f}% | Val: {100*v_correct/v_total:.2f}%")